## HW 4

In [1]:
# As always, we start with our favorite standard imports. 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA


## Homework 4 Spring 2026
- 6.6.0 (2+2+2+2+2+2= 12 points) Conceptual questions
  - (a) What is the definition of scale equivariant?
  -  (b) Why is it important to standardize the predictors when using ridge regression or the lasso?
  -  (c) What is the difference between ridge regression and the lasso?
  -  (d) What is an advantage of using ridge regression or the lasso over least squares linear regression?
  -  (e) What is the purpose of PCA?
  -  (f) What does the first principle component maximize?
    
- 6.6.9 (a-g) In this exercise, we will predict the number of applications received
using the other variables in the `College` data set.
  


### Grading distribution 

- 6.60 (12 points)
- 6.6.9 (44 points)


## 6.6.0 Conceptual questions
  - (a) What is the definition of scale equivariant?

### YOUR ANSWER HERE ###

A method is scale equivariant if multiplying a predictor by a constant causes a corresponding, predictable change in the estimate, so the method’s conclusions do not depend on the units of measurement.

-  (b) Why is it important to standardize the predictors when using ridge regression or the lasso?


### YOUR ANSWER HERE ###
It is important to standardize predictors because ridge regression and the lasso penalize the size of coefficients. Without standardization, predictors measured on larger scales would be penalized more heavily than those on smaller scales, leading to unfair and misleading shrinkage.

-  (c) What is the difference between ridge regression and the lasso?
 


### YOUR ANSWER HERE ###

Ridge regression uses an L2 penalty and shrinks coefficients toward zero but never sets them exactly to zero, while the lasso uses an L1 penalty and can set some coefficients exactly to zero, performing variable selection.

 -  (d) What is an advantage of using ridge regression or the lasso over least squares linear regression?
  

### YOUR ANSWER HERE ###

Ridge regression and the lasso reduce overfitting and improve prediction accuracy when predictors are highly correlated or when there are many predictors, unlike ordinary least squares, which can have high variance in these cases.

-  (e) What is the purpose of PCA (Principal Component Analysis)?


### YOUR ANSWER HERE ###

The purpose of PCA is to reduce the dimensionality of a dataset by transforming correlated variables into a smaller set of uncorrelated components that capture most of the variance.

-  (f) What does the first principle component maximize?

### YOUR ANSWER HERE ###

The first principal component maximizes the variance of the projected data, capturing as much information from the original variables as possible in a single component.

## 6.6.9
In this exercise, we will predict the number of applications received
using the other variables in the `College` data set.

In [5]:
## Load the dataset
url = "https://msu-cmse-courses.github.io/CMSE381-S26/_downloads/cc29ec6408d657de88bc7fe6de6b1170/College.csv"
college_df = pd.read_csv(url)
college_df = college_df.set_index('Unnamed: 0')

## One-hot encode the categorical variable
college_df = pd.get_dummies(college_df, drop_first=True)

college_df.head()

,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate,Private_Yes
Unnamed: 0,,,,,,,,,,,,,,,,,,
Abilene Christian University,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60,True
Adelphi University,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56,True
Adrian College,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54,True
Agnes Scott College,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59,True
Alaska Pacific University,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15,True


In [6]:
# Convert entire dataframe to float before modeling
college_df = college_df.astype(float)

print(college_df.dtypes)

Apps           float64
Accept         float64
Enroll         float64
Top10perc      float64
Top25perc      float64
F.Undergrad    float64
P.Undergrad    float64
Outstate       float64
Room.Board     float64
Books          float64
Personal       float64
PhD            float64
Terminal       float64
S.F.Ratio      float64
perc.alumni    float64
Expend         float64
Grad.Rate      float64
Private_Yes    float64
dtype: object


(a) (5 points):Split the data in test set and training set (this will only be used for the linear regression, all other models are cross-validated with the whole set).

In [7]:
###YOUR CODE HERE###

X = college_df.drop("Apps", axis=1)
y = college_df["Apps"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (621, 17)
X_test shape: (156, 17)
y_train shape: (621,)
y_test shape: (156,)


In [8]:
print(X_train.dtypes)

Accept         float64
Enroll         float64
Top10perc      float64
Top25perc      float64
F.Undergrad    float64
P.Undergrad    float64
Outstate       float64
Room.Board     float64
Books          float64
Personal       float64
PhD            float64
Terminal       float64
S.F.Ratio      float64
perc.alumni    float64
Expend         float64
Grad.Rate      float64
Private_Yes    float64
dtype: object


(b) (5 points): Fit a linear model using least squares on the training set, and
report the test error obtained.

In [27]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lm = LinearRegression()
lm.fit(X_train, y_train)
y_pred = lm.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Linear Regression Test RMSE:", test_rmse)

Linear Regression Test RMSE: 1221.6559986506193


#### ✅ Question (b): Documenting Your Solution Process (3 points)

Please answer the following clearly and completely:

1. **Prior Knowledge vs. External Resources (1 point)**  
   Indicate which parts of Question (b) you completed using your own prior knowledge, and which parts you completed using external resources (e.g., generative AI, past assignments, Stack Overflow, Google, etc.).

### YOUR ANSWER HERE ###

I was already familiar with the concepts behind linear regression, training/test splits, and RMSE from prior coursework, so I was able to plan the analysis and interpret the results using my own knowledge. I needed some help with the specific Python syntax for splitting the data, fitting the linear model, and calculating the test RMSE, for which I consulted external resources including documentation and coding examples.

2. **Required Documentation (2 points)**  
   - For any part where you used **generative AI**, you must include the exact prompts you entered and the corresponding AI outputs. **Copy and paste them directly**.  
   - For any part where you used other external resources, list those sources.  
   - For parts completed without external resources, briefly state what prior knowledge you relied on (no detailed explanation required).

Responses that do not include **prompts and AI outputs** (when applicable) will not receive full credit.


In [10]:
### YOUR ANSWER HERE##

# Other External Resources: None.
# Prior Knowledge: I relied on my prior knowledge of linear regression, RMSE, and the idea of splitting data into training and test sets to understand what needed to be done.

#YOUR PROMPTS##

# “Help me split the College dataset into training and test sets for linear regression.”

##AI OUTPUTS##

# Separate predictors and target
# X = college_df.drop("Apps", axis=1)
# y = college_df["Apps"]

# This was used to get a skeleton code for splitting the data into training and test sets.

(c) (5 points): Fit a ridge regression model on the training set, with λ chosen by cross-validation. Report the test error obtained.

In [28]:
###YOUR CODE HERE###

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

alphas = np.logspace(-3, 3, 100)
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', RidgeCV(alphas=alphas, cv=10))])
ridge_pipeline.fit(X_train, y_train)

best_alpha = ridge_pipeline.named_steps['ridge'].alpha_
print("Best lambda (Ridge):", best_alpha)
y_pred = ridge_pipeline.predict(X_test)

ridge_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Ridge Test RMSE:", ridge_rmse)

Best lambda (Ridge): 4.9770235643321135
Ridge Test RMSE: 1196.5482511210803


(d)(5 points): Fit a lasso model on the training set, with λ chosen by cross-
validation. Report the test error obtained, along with the num-
ber of non-zero coefficient estimates.

In [38]:
###YOUR CODE HERE###

from sklearn.linear_model import LassoCV

lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(alphas=None, cv=10, max_iter=10000, random_state=42))])

lasso_pipeline.fit(X_train, y_train)

best_alpha_lasso = lasso_pipeline.named_steps['lasso'].alpha_
print("Best lambda (Lasso):", best_alpha_lasso)

y_pred = lasso_pipeline.predict(X_test)
lasso_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Lasso Test RMSE:", lasso_rmse)

lasso_model = lasso_pipeline.named_steps['lasso']
nonzero = np.sum(lasso_model.coef_ != 0)
print("Number of non-zero coefficients:", nonzero)

# Lasso Test RMSE: 1216.1157829968909
# Number of non-zero coefficients: 17

Best lambda (Lasso): 3.721117921122065
Lasso Test RMSE: 1216.1157829968909
Number of non-zero coefficients: 17


/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


#### ✅ Question (d): Documenting Your Solution Process (3 points)

Please answer the following clearly and completely:

1. **Prior Knowledge vs. External Resources (1 point)**  
   Indicate which parts of Question (d) you completed using your own prior knowledge, and which parts you completed using external resources (e.g., generative AI, past assignments, Stack Overflow, Google, etc.).

### YOUR ANSWER HERE ###

I was already familiar with the concepts behind Lasso regression, cross-validation, test RMSE, and counting non-zero coefficients, so I understood what the analysis required. I needed help with the specific Python syntax to implement LassoCV, select the best lambda, make predictions, and count non-zero coefficients, for which I used generative AI.

2. **Required Documentation (2 points)**  
   - For any part where you used **generative AI**, you must include the exact prompts you entered and the corresponding AI outputs. **Copy and paste them directly**.  
   - For any part where you used other external resources, list those sources.  
   - For parts completed without external resources, briefly state what prior knowledge you relied on (no detailed explanation required).

Responses that do not include **prompts and AI outputs** (when applicable) will not receive full credit.


In [34]:
### YOUR ANSWER HERE##

# Other External Resources: None.
# Prior Knowledge: I used my understanding of Lasso regression, cross-validation, regularization (lambda selection),
# and how to evaluate model performance using test RMSE. I also knew that Lasso can shrink some coefficients to zero,
# so choosing the tuning parameter carefully is important.

#YOUR PROMPTS##

# “Help me fit a Lasso regression model using cross-validation on the training data. I want to select the best value of
# lambda (alpha), report the test RMSE, and define a proper grid of alpha values to avoid convergence warnings.”

##AI OUTPUTS##

# Define a grid of alpha values (lambda)
# alphas = np.logspace(-3, 3, 100)

# lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=42)
# lasso_cv.fit(X_train, y_train)

# This was used to get working code for fitting LassoCV, computing test RMSE, and setting the alphas grid correctly.

(e)(5 points): Fit a PCR model on the training set, with M chosen by cross-
validation. Report the test error obtained, along with the value
of M selected by cross-validation.

In [35]:
###YOUR CODE HERE###

from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

max_components = X_train.shape[1]
rmse_scores = []

for m in range(1, max_components + 1):
    pcr_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=m)),
        ('lm', LinearRegression())])
    
    scores = cross_val_score(pcr_pipeline, X_train, y_train, cv=10, scoring='neg_mean_squared_error')
    rmse = np.sqrt(-scores.mean())
    rmse_scores.append(rmse)

best_m = np.argmin(rmse_scores) + 1
print("Best number of components (PCR):", best_m)

final_pcr = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=best_m)), ('lm', LinearRegression())])
final_pcr.fit(X_train, y_train)

y_pred = final_pcr.predict(X_test)
pcr_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("PCR Test RMSE:", pcr_rmse)

# PCR Test RMSE: 1221.6559986506113
# Best number of components (PCR): 17

Best number of components (PCR): 17
PCR Test RMSE: 1221.6559986506113


(f) (5 points): Fit a PLS model on the training set, with M chosen by cross-
validation. Report the test error obtained, along with the value
of M selected by cross-validation.

In [36]:
###YOUR CODE HERE###

from sklearn.cross_decomposition import PLSRegression

max_components = min(X_train.shape[1], 20)
rmse_scores = []

for m in range(1, max_components + 1):
    pls = PLSRegression(n_components=m)
    scores = cross_val_score(pls, X_train, y_train, cv=10, scoring='neg_mean_squared_error')
    rmse = np.sqrt(-scores.mean())
    rmse_scores.append(rmse)
best_m_pls = np.argmin(rmse_scores) + 1
print("Best number of components (PLS):", best_m_pls)
final_pls = PLSRegression(n_components=best_m_pls)
final_pls.fit(X_train, y_train)

y_pred = final_pls.predict(X_test)
pls_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("PLS Test RMSE:", pls_rmse)

# PLS Test RMSE: 1218.1216202997332
# Best number of components (PLS): 13

Best number of components (PLS): 13
PLS Test RMSE: 1218.1216202997332


#### ✅ Question (f): Documenting Your Solution Process (3 points)

Please answer the following clearly and completely:

1. **Prior Knowledge vs. External Resources (1 point)**  
   Indicate which parts of Question (f) you completed using your own prior knowledge, and which parts you completed using external resources (e.g., generative AI, past assignments, Stack Overflow, Google, etc.).

### YOUR ANSWER HERE ###

I was already familiar with the concepts behind PCR and PLS regression, cross-validation for selecting the number of components, and computing test RMSE, so I understood what the analysis required. I needed help with the specific Python implementation for looping over components, performing cross-validation, and computing the test RMSE using sklearn, for which I used generative AI.

2. **Required Documentation (2 points)**  
   - For any part where you used **generative AI**, you must include the exact prompts you entered and the corresponding AI outputs. **Copy and paste them directly**.  
   - For any part where you used other external resources, list those sources.  
   - For parts completed without external resources, briefly state what prior knowledge you relied on (no detailed explanation required).

Responses that do not include **prompts and AI outputs** (when applicable) will not receive full credit.


In [33]:
### YOUR ANSWER HERE##

# Other External Resources: None.
# Prior Knowledge: I relied on my understanding of principal component regression (PCR), partial least squares (PLS),
# cross-validation, and test RMSE. I used this background knowledge to structure the modeling steps correctly
# (train/test split first, then cross-validation only on the training set) and to interpret the final model comparison results.

#YOUR PROMPTS##

# “Help me fit a PCR model on the training set, choose the number of components M by cross-validation,
# and report the test RMSE and selected M.”

##AI OUTPUTS##

#The AI suggested looping over possible values of M fitting a pipeline with scaling, PCA, and linear regression,
# and using cross-validation to compute the RMSE for each value. The structure was:

# n_components = list(range(1, X_train.shape[1]+1))
# best_rmse = np.inf
# best_M = None
# best_model = None

# for m in n_components:
#    pcr = Pipeline([('pca', PCA(n_components=m)), ('lr', LinearRegression())])

(g) (5 points): Comment on the results obtained. How accurately can we pre-
dict the number of college applications received? Is there much
difference among the test errors resulting from these five ap-
proaches?

### YOUR COMMENT HERE ###

The test RMSE values are all between about 1196 and 1222 applications. Since RMSE tells us the typical prediction error, this means our predictions are usually off by around 1,200 applications.

Whether that is “accurate” depends on how many applications schools usually receive. If colleges are getting several thousand applications, then being off by about 1,200 means the models are doing a decent job, but they are not extremely precise. There is still a good amount of variation that the models cannot fully explain.

There is not a big difference between the five methods. Ridge regression does the best, but only by a small amount. Lasso and PLS are very close to regular linear regression, and PCR gives almost the exact same result as linear regression.

Overall, all five approaches perform pretty similarly. Ridge helps a little, but none of the methods dramatically improve prediction accuracy compared to the others.